# Clean and format data for general use: REDS-Index and REDS-Recall
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import defaultdict

import numpy as np
import pandas as pd

from hk1_r12ter_analysis.config import (
    GENE_ID,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
    logger,
)
from hk1_r12ter_analysis.io import (
    load_dataframe_from_file,
    make_filename,
    save_dataframe_to_file,
)

## Clean and format REDS Index data

In [ ]:
source_type = "REDS-Index"
sample_key = "INDEX DONOR ID"
group_key = None
# Directory paths
dirpath = RAW_DATA_DIR / "REDS"
dirpath.mkdir(parents=True, exist_ok=True)

### REDS Index Metadata
#### Load data

In [ ]:
filename_args = (source_type, "Donor", "Metadata")
logger.debug(f"Cleaning and formatting data for '{'-'.join(filename_args)}'...")
filename = make_filename(*filename_args)
df = load_dataframe_from_file(dirpath / filename)
df = df.dropna(subset=sample_key)

column_label = "Ethnicity"
replacement_values = {"CAUCASIAN_OTHER": "CAUCASIAN/OTHER", "HIGH": np.nan}
logger.debug(f"Replacing values in '{column_label}' column...")
df[column_label] = df[column_label].replace(replacement_values)
logger.info(f"Replaced values in '{column_label}' column.")
df = df.set_index(sample_key)
df = df.sort_index(axis=0)
logger.info(f"Cleaned and formatted data for '{'-'.join(filename_args)}'.")

df_metadata_index = df.copy()
df_metadata_index

### REDS Index Genomic data
#### Load data
##### Split data for annotation information

In [ ]:
filename_args = (source_type, "Donor", f"Genomics-{GENE_ID}")
filename = make_filename(*filename_args)

df_genomics_all = load_dataframe_from_file(dirpath / filename, index_col=0)
df_genomics_all.index.name = "ID"

df_genomics_all["RSID"] = df_genomics_all.index.tolist()
df_genomics_all["RSID"] = df_genomics_all["RSID"].str.extract(r"^(\S+rs\d+)")

df_genomics_all["RSID"] = df_genomics_all["RSID"].fillna(
    dict(zip(df_genomics_all.index.tolist(), df_genomics_all.index.tolist()))
)
df_genomics_annotations = df_genomics_all.loc[:, ~df_genomics_all.columns.str.startswith("RO0")]
df_genomics_annotations

#### Parse genomic annotations
##### Datatype format for annotations:
* <ID=ANN,Number=.,Type=String,Description="Functional annotations: 'Allele | Annotation | Annotation_Impact | Gene_Name | GENE_ID | Feature_Type | Feature_ID | Transcript_BioType | Rank | HGVS.c | HGVS.p | cDNA.pos / cDNA.length | CDS.pos / CDS.length | AA.pos / AA.length | Distance | ERRORS / WARNINGS / INFO' ">
* <ID=LOF,Number=.,Type=String,Description="Predicted loss of function effects for this variant. Format: ">
* <ID=NMD,Number=.,Type=String,Description="Predicted nonsense mediated decay effects for this variant. Format: 'Gene_Name | GENE_ID | Number_of_transcripts_in_gene | Percent_of_transcripts_affected'">

In [ ]:
str_formats = {
    "ANN": "Allele | Annotation | Annotation_Impact | Gene_Name | GENE_ID | Feature_Type | Feature_ID | Transcript_BioType | Rank | HGVS.c | HGVS.p | cDNA.pos / cDNA.length | CDS.pos / CDS.length | AA.pos / AA.length | Distance | ERRORS / WARNINGS / INFO",
    "LOF": "Gene_Name | GENE_ID | Number_of_transcripts_in_gene | Percent_of_transcripts_affected",
    "NMD": "Gene_Name | GENE_ID | Number_of_transcripts_in_gene | Percent_of_transcripts_affected",
}

column_labels = {
    key: [col_name.strip() for col_name in str_format.split(" | ")]
    for key, str_format in str_formats.items()
}
series_info_column = df_genomics_annotations["INFO"]
columns = (
    pd.concat(
        (
            series_info_column.str.extract("(?P<ANN>ANN)"),
            series_info_column.str.extract("(?P<LOF>LOF)"),
            series_info_column.str.extract("(?P<NMD>NMD)"),
        ),
        axis=1,
    )
    .dropna(axis=1, how="all")
    .columns
)
df_genomics_info_parsed_all = series_info_column.str.split(";", expand=True).rename(
    dict(list(enumerate(columns))), axis=1
)
for col, df in df_genomics_info_parsed_all.items():
    df = df.str.removeprefix(f"{col}=")
    if col != "ANN":
        df = df.str.lstrip("(")
        df = df.str.rstrip(")")
    else:
        df = df.str.split(",", expand=True)
        df = df.melt(ignore_index=False, var_name="ANN_Priority", value_name=col)
        df.set_index(["ANN_Priority"], append=True, inplace=True)
    df = df.sort_index().squeeze()
    df = df.str.split("|", expand=True)
    df.columns = [f"{col}_{x}" for x in column_labels[col]]
    df_genomics_info_parsed_all = df.join(df_genomics_info_parsed_all, how="left")
    df_genomics_info_parsed_all = df_genomics_info_parsed_all.drop(col, axis=1)
df_genomics_info_parsed_all = df_genomics_annotations.join(df_genomics_info_parsed_all, how="left")
df_genomics_info_parsed_all = df_genomics_info_parsed_all[
    df_genomics_info_parsed_all["ANN_Gene_Name"] == GENE_ID
].copy()
df_genomics_info_parsed_all = df_genomics_info_parsed_all.drop_duplicates(
    subset=df_genomics_info_parsed_all.columns[:7]
)
df_genomics_info_parsed_all = df_genomics_info_parsed_all.drop("INFO", axis=1).fillna("")
df_genomics_info_parsed_all

#### Create DataFrame for allele frequencies

In [ ]:
df_genomics_index_all = df_genomics_all.loc[:, df_genomics_all.columns.str.startswith("RO0")].T
df_genomics_index_all.index.name = sample_key
df_genomics_index_all.columns.name = None
df_genomics_index_all = df_genomics_index_all.replace(-1, pd.NA)
df_genomics_index_all = df_genomics_index_all.sort_index(axis=0)
df_genomics_index_all

In [ ]:
df_alleles_index_all = df_genomics_index_all.apply(pd.to_numeric, errors="coerce")
# Calculate frequency of NA values
snps = df_alleles_index_all.columns
df_NA = pd.DataFrame.from_dict(
    {
        snp: df_alleles_index_all.loc[:, snp]
        .value_counts(dropna=False, normalize=True)
        .get(pd.NA, 0)
        for snp in snps
    },
    orient="index",
    columns=["NA"],
)
# Calculate genotype frequencies with NA values dropped
df_alleles_index_all = pd.concat(
    [df_alleles_index_all.loc[:, snp].value_counts(dropna=True, normalize=True) for snp in snps],
    axis=1,
)
df_alleles_index_all.columns = snps
df_alleles_index_all = df_alleles_index_all.rename(
    {0: "HMOZ_REF", 1: "HTRZ", 2: "HMOZ_ALT"}, axis=0
)
# Get allele frequencies
df_alleles_index_all.loc["AF_REF"] = (
    df_alleles_index_all.loc["HMOZ_REF"] + df_alleles_index_all.loc["HTRZ"] / 2
)
df_alleles_index_all.loc["AF_ALT"] = (
    df_alleles_index_all.loc["HMOZ_ALT"] + df_alleles_index_all.loc["HTRZ"] / 2
)
df_alleles_index_all = df_alleles_index_all.T
df_alleles_index_all = df_alleles_index_all.merge(
    df_NA, left_index=True, right_index=True, how="left"
)
df_alleles_index_all = df_alleles_index_all.fillna(0).astype(float)
df_alleles_index_all.index.name = "ID"
df_alleles_index_all

#### Filter SNPs by missingness, minimum MAF value, and homozygous (HMOZ) percentages

In [ ]:
# Filter by missingness per SNP
NA_per_SNP_percent = 1
minimum_MAF_percent = 0.01
minimum_HMOZ_percent = 0
df_alleles_index = df_alleles_index_all.copy()

if "NA" in df_alleles_index.columns:
    df_alleles_index = df_alleles_index[df_alleles_index["NA"] < NA_per_SNP_percent]
df_alleles_index = df_alleles_index[
    (minimum_MAF_percent <= df_alleles_index[["AF_REF", "AF_ALT"]]).all(axis=1)
]
df_alleles_index = df_alleles_index[
    (minimum_HMOZ_percent <= df_alleles_index[["HMOZ_REF", "HMOZ_ALT"]]).all(axis=1)
]
df_alleles_index.index.names = ["ID"]
# Filter genomics using allele frequencies
df_genomics_index = df_genomics_index_all.loc[:, df_alleles_index.index.tolist()].sort_index(
    axis=1
)
df_genomics_info_parsed = df_genomics_info_parsed_all.loc[
    df_alleles_index.index.tolist()
].sort_index(axis=0)
df_alleles_index

### Load REDS Index Proteomics
#### Load Protein Metadata

In [ ]:
filename_args = (source_type, "RBC", "Proteomics", "Metadata")
logger.debug(f"Cleaning and formatting data for '{'-'.join(filename_args)}'...")
filename = make_filename(*filename_args)
df = load_dataframe_from_file(dirpath / filename)
# Check to see if expected columns are included. If so, then order columns as listed.
# Comes directly from UniProt if possible
df = df.loc[
    :,
    [
        "Entry",
        "Entry Name",
        "Protein",
        "Protein Names",
        "Gene Names (Primary)",
        "Length",
        "Mass",  # Should be in DA
    ],
]
# Sort the data via alphabetical order of protein IDs for consistency
mapping_dict = {"Entry": "UniProt", "Gene Names (Primary)": "Gene"}
df = df.rename(mapping_dict, axis=1).sort_values(by="UniProt").reset_index(drop=True)
logger.info(f"Cleaned and formatted data for '{'-'.join(filename_args)}'.")
df_proteomics_meta_index = df.copy()
df_proteomics_meta_index

#### Load proteomics

In [ ]:
filename_args = (source_type, "Donor", "RBC", "Proteomics")
logger.debug(f"Cleaning and formatting '{'-'.join(filename_args)}'...")
filename = make_filename(*filename_args)
df = load_dataframe_from_file(
    dirpath / filename, index_col=[key for key in [sample_key, group_key] if key]
)

old_id_type = "Protein"
new_id_type = "Protein"
if old_id_type != new_id_type:
    mapping_dict = df_proteomics_meta_index.set_index(old_id_type)[new_id_type].to_dict()
    df = df.rename(mapping_dict, axis=1)
df = df.sort_index(axis=0).sort_index(axis=1)

# Combine duplicate proteins (isoforms) through summation
df = df.rename({x: x.split(".")[0] for x in df.columns}, axis=1)
df = df.T.groupby(level=0).sum().T

# Remove duplicates
duplicates = list(df.loc[df.index.duplicated()].index)
df = df.loc[~df.index.isin(duplicates)].copy()
df = df.sort_index(axis=0).sort_index(axis=1)
logger.info(f"Cleaned and formatted '{'-'.join(filename_args)}'.")
df_proteomics_index = df.copy()
df_proteomics_index

### Load REDS Index Metabolomics

In [ ]:
filename_args = (source_type, "Donor", "RBC", "Metabolomics")
logger.debug(f"Cleaning and formatting '{'-'.join(filename_args)}'...")
filename = make_filename(*filename_args)
df = load_dataframe_from_file(
    dirpath / filename, index_col=[key for key in [sample_key, group_key] if key]
)

# TODO Plate Normalization for raw data
# df = df.drop("plate", axis=1)

# Already normalized by plate and median
df = df.sort_index(axis=0).sort_index(axis=1)
logger.info(f"Cleaned and formatted '{'-'.join(filename_args)}'.")
df_metabolomics_index = df.copy()
df_metabolomics_index

### Load REDS Index mQTL Data

In [ ]:
filename_args = (source_type, f"Genomics-{GENE_ID}", "mQTL")
logger.debug(f"Cleaning and formatting data for '{'-'.join(filename_args)}'...")
filename = make_filename(*filename_args)
df = load_dataframe_from_file(dirpath / filename)
# Update column names while fixing spreadsheet column name issue
df = df.rename(
    {"Location_b38": "p-value", "BETA": "Beta", "POS": "Position", "POS_b37": "Chromosome"}, axis=1
)
df["-log10(p-value)"] = df["p-value"].apply(lambda x: -np.log10(x))
cols_to_keep = [
    "rsid",
    "Metabolite",
    "Chromosome",
    "Position",
    "p-value",
    "-log10(p-value)",
    "Beta",
    "SE",
]
df = df[cols_to_keep].drop_duplicates()
df = df.sort_values(by=["rsid", "p-value"]).reset_index(drop=True)

rsids = df_genomics_index.columns
rsids = pd.DataFrame.from_dict(dict(zip(rsids, rsids.str.extract(r"(rs\d+)")[0])), orient="index")
rsids = rsids.dropna().reset_index(drop=False).set_index(0)
rsids = rsids.squeeze().to_dict()
df = df[df["rsid"].isin(rsids.keys())].reset_index(drop=True).copy()
df["ID"] = df["rsid"].replace(rsids)

replacement_values = {
    "Sedoheptulose7phosphate": "Sedoheptulose 7-phosphate",
    "gammaGlutamylSemethylselenocysteine": "gamma-Glutamyl-Se-methylselenocysteine",
    "Derythrose_4phosphate": "D-Erythrose 4-phosphate",
    "DHexosephosphate": "D-Hexose phosphate",
    "X5Oxoproline": "5-Oxoproline",
    "X6PhosphoDgluconate": "6-Phospho-D-gluconate",
    "X56Indolequinone2carboxylic_acid": "5,6-Indolequinone 2-carboxylic acid",
    "DFructose16bisphosphate": "D-Fructose 1,6-bisphosphate",
    "UDPglucose": "UDP-glucose",
    "glycine": "Glycine",
    "Pentosephosphates": "Pentose phosphates",
    "Lcysteine": "L-cysteine",
    "NAcetylneuraminate": "N-Acetylneuraminate",
    "X3prime_5primeCyclicAMP": "3',5'-Cyclic AMP",
    "gammaLGlutamylLcysteine": "gamma-L-Glutamyl-L-cysteine",
    "X2Hydroxyglutarate": "2-Hydroxyglutarate",
    "SGlutathionylLcysteine": "S-Glutathionyl-L-cysteine",
    "Ribose_Ribulose_Arabinose_Xylose_Xylulose": "Pentoses",
    "Acetylphosphate": "Acetyl phosphate",
    "ITP": "ITP",
    "Indole": "Indole",
    "Lactate": "Lactate",
    "Diphosphate": "Diphosphate",
    "Malate": "Malate",
    "Fumarate": "Fumarate",
    "Hypoxanthine": "Hypoxanthine",
    "Spermine": "Spermine",
}
df["Metabolite"] = df["Metabolite"].replace(replacement_values)
logger.info(f"Cleaned and formatted data for '{'-'.join(filename_args)}'.")
df = df.loc[:, ["ID"] + cols_to_keep]
df = df.rename({"rsid": "RSID"}, axis=1)
df = df.set_index(["ID", "RSID", "Metabolite"]).sort_index()
df_mQTL_index = df.copy()
df_mQTL_index

## Load REDS Recall data

In [ ]:
source_type = "REDS-Recall"
sample_key = "RECALL DONOR ID"
group_key = "Day"

### Load REDS Recall Metadata

In [ ]:
filename_args = (source_type, "Donor", "Metadata")
logger.debug(f"Cleaning and formatting data for '{'-'.join(filename_args)}'...")
filename = make_filename(*filename_args)
df = load_dataframe_from_file(dirpath / filename)
df = df.dropna(subset=sample_key)

column_label = "Ethnicity"
replacement_values = {"CAUCASIAN_OTHER": "CAUCASIAN", "HIGH": np.nan}
logger.debug(f"Replacing values in '{column_label}' column...")
df[column_label] = df[column_label].replace(replacement_values)
logger.info(f"Replaced values in '{column_label}' column.")
df = df.set_index(sample_key)
df = df.rename({x: x.replace("Transfer", "Adjusted") for x in df.columns}, axis=1)
df = df.sort_index(axis=0)
logger.info(f"Cleaned and formatted data for '{'-'.join(filename_args)}'.")

df_metadata_recall = df.copy()
df_metadata_recall

### REDS Recall Genomics
#### Map genomics using REDS Index Donor Recall mapping

In [ ]:
filename_args = ("REDS", "Index", "Recall", "Donor", "Mapping")
filename = make_filename(*filename_args)
df_donor_mapping = load_dataframe_from_file(dirpath / filename)
logger.debug("Mapping REDS Index IDs to REDS Recall IDs...")

df_genomics_recall = df_donor_mapping.set_index(df_donor_mapping.columns[0]).join(
    df_genomics_index, how="left"
)
df_genomics_recall = df_genomics_recall.set_index(sample_key)
df_genomics_recall = df_genomics_recall.convert_dtypes()
df_genomics_recall = df_genomics_recall.dropna(how="all", axis=0)
df_genomics_recall = df_genomics_recall.sort_index(axis=0)
logger.info("Mapped REDS Index IDs to REDS Recall IDs.")
df_genomics_recall

### Load REDS Recall Proteomics
#### Load Protein Metadata

In [ ]:
filename_args = (source_type, "RBC", "Proteomics", "Metadata")
logger.debug(f"Cleaning and formatting data for '{'-'.join(filename_args)}'...")
filename = make_filename(*filename_args)
df = load_dataframe_from_file(dirpath / filename)
# Check to see if expected columns are included. If so, then order columns as listed.
# Comes directly from UniProt if possible
df = df.loc[
    :,
    [
        "Entry",
        "Entry Name",
        "Protein",
        "Protein Names",
        "Gene Names (Primary)",
        "Length",
        "Mass",  # Should be in DA
    ],
]
# Sort the data via alphabetical order of protein IDs for consistency
mapping_dict = {"Entry": "UniProt", "Gene Names (Primary)": "Gene"}
df = df.rename(mapping_dict, axis=1).sort_values(by="UniProt").reset_index(drop=True)
logger.info(f"Cleaned and formatted data for '{'-'.join(filename_args)}'.")
df_proteomics_meta_recall = df.copy()
df_proteomics_meta_recall

#### Load Proteomics

In [ ]:
# Data arguments
filename_args = (source_type, "Donor", "RBC", "Proteomics")
logger.debug(f"Cleaning and formatting data for '{'-'.join(filename_args)}'...")
filename = make_filename(*filename_args)
df = load_dataframe_from_file(
    dirpath / filename, index_col=[key for key in [sample_key, group_key] if key]
)

old_id_type = "Protein"
new_id_type = "Protein"
if old_id_type != new_id_type:
    mapping_dict = df_proteomics_meta_index.set_index(old_id_type)[new_id_type].to_dict()
    df = df.rename(mapping_dict, axis=1)
df = df.sort_index(axis=0).sort_index(axis=1)

# Combine duplicate proteins (isoforms) through summation
df = df.rename({x: x.split(".")[0] for x in df.columns}, axis=1)
df = df.T.groupby(level=0).sum().T

# Remove duplicate samples
duplicates = list(df.loc[df.index.duplicated()].index)
df = df.loc[~df.index.isin(duplicates)].copy()
# df = df[df.index.isin(df_genomics_recall.index, level=sample_key)].copy()
df = df.sort_index(axis=0).sort_index(axis=1)
logger.info(f"Cleaned and formatted data for '{'-'.join(filename_args)}'.")
df_proteomics_recall = df.copy()
df_proteomics_recall

### Load REDS Recall Metabolomics

In [ ]:
filename_args = (source_type, "Donor", "RBC", "Metabolomics")
logger.debug(f"Cleaning and formatting data for '{'-'.join(filename_args)}'...")
filename = make_filename(*filename_args)
df = load_dataframe_from_file(
    dirpath / filename, index_col=[key for key in [sample_key, group_key] if key]
)

# Remove duplicates
duplicates = list(df.loc[df.index.duplicated()].index)
df = df.loc[~df.index.isin(duplicates)].copy()
# df = df[df.index.isin(df.index, level=sample_key)].copy()
df = df.sort_index(axis=0).sort_index(axis=1)
logger.info(f"Cleaned and formatted data for '{'-'.join(filename_args)}'.")
df_metabolomics_recall = df.copy()
df_metabolomics_recall

### Load REDS Recall Lipidomics

In [ ]:
filename_args = (source_type, "Donor", "RBC", "Lipidomics")
logger.debug(f"Cleaning and formatting data for '{'-'.join(filename_args)}'...")
filename = make_filename(*filename_args)
df = load_dataframe_from_file(
    dirpath / filename, index_col=[key for key in [sample_key, group_key] if key]
)
# Remove duplicates
duplicates = list(df.loc[df.index.duplicated()].index)
df = df.loc[~df.index.isin(duplicates)].copy()
df = df.sort_index(axis=0).sort_index(axis=1)
logger.info(f"Cleaned and formatted data for '{'-'.join(filename_args)}'.")
df_lipidomics_recall = df.copy()
df_lipidomics_recall

In [ ]:
filename_args = (source_type, "Donor", "RBC", "Lipidomics", "DegUnsat")
logger.debug(f"Cleaning and formatting data for '{'-'.join(filename_args)}'...")
filename = make_filename(*filename_args)
df = load_dataframe_from_file(
    dirpath / filename, index_col=[key for key in [sample_key, group_key] if key]
)
# Remove duplicates
duplicates = list(df.loc[df.index.duplicated()].index)
df = df.loc[~df.index.isin(duplicates)].copy()
# df = df[df.index.isin(df.index, level=sample_key)].copy()
df = df.sort_index(axis=0).sort_index(axis=1)
logger.info(f"Cleaned and formatted data for '{'-'.join(filename_args)}'.")
df_lipid_degunsat_recall = df.copy()
df_lipid_degunsat_recall

In [ ]:
filename_args = (source_type, "Donor", "RBC", "Lipidomics", "LipidClass")
logger.debug(f"Cleaning and formatting data for '{'-'.join(filename_args)}'...")
filename = make_filename(*filename_args)
df = load_dataframe_from_file(
    dirpath / filename, index_col=[key for key in [sample_key, group_key] if key]
)
# Remove duplicates
duplicates = list(df.loc[df.index.duplicated()].index)
df = df.loc[~df.index.isin(duplicates)].copy()
# df = df[df.index.isin(df.index, level=sample_key)].copy()
df = df.sort_index(axis=0).sort_index(axis=1)
logger.info(f"Cleaned and formatted data for '{'-'.join(filename_args)}'.")
df_lipid_classes_recall = df.copy()
df_lipid_classes_recall

### Load REDS Recall Vesicles and mtDNA

In [ ]:
filename_args = (source_type, "Donor", "RBC", "Vesicles", "mtDNA")
logger.debug(f"Cleaning and formatting data for '{'-'.join(filename_args)}'...")
filename = make_filename(*filename_args)
df = load_dataframe_from_file(
    dirpath / filename, index_col=[key for key in [sample_key, group_key] if key]
)
# Remove duplicates
duplicates = list(df.loc[df.index.duplicated()].index)
df = df.loc[~df.index.isin(duplicates)].copy()
# df = df[df.index.isin(df.index, level=sample_key)].copy()
df = df.sort_index(axis=0).sort_index(axis=1)
logger.info(f"Cleaned and formatted data for '{'-'.join(filename_args)}'.")
df_vescicles_mtDNA_recall = df.copy()
df_vescicles_mtDNA_recall

### Load REDS Recall Trace Element Analysis

In [ ]:
filename_args = (source_type, "Donor", "RBC", "TraceElements")
logger.debug(f"Cleaning and formatting data for '{'-'.join(filename_args)}'...")
filename = make_filename(*filename_args)
df = load_dataframe_from_file(
    dirpath / filename, index_col=[key for key in [sample_key, group_key] if key]
)
# Remove duplicates
duplicates = list(df.loc[df.index.duplicated()].index)
df = df.loc[~df.index.isin(duplicates)].copy()
# df = df[df.index.isin(df.index, level=sample_key)].copy()
df = df.sort_index(axis=0).sort_index(axis=1)
logger.info(f"Cleaned and formatted data for '{'-'.join(filename_args)}'.")
df_trace_elem_recall = df.copy()
df_trace_elem_recall

### Export REDS Data

In [ ]:
output_filenames_dfs = defaultdict(dict)
output_dirpath = PROCESSED_DATA_DIR / "REDS"
output_dirpath.mkdir(parents=True, exist_ok=True)
output_filenames_dfs[output_dirpath].update(
    {
        make_filename("REDS", "Index", f"Genomics-{GENE_ID}", "mQTL"): df_mQTL_index,
        make_filename("REDS", f"Genomics-{GENE_ID}", "Metadata"): df_genomics_info_parsed,
    }
)

output_dirpath = INTERIM_DATA_DIR / "REDS"
output_dirpath.mkdir(parents=True, exist_ok=True)
output_filenames_dfs[output_dirpath].update(
    {
        # REDS Genomics
        # make_filename("REDS", f"Genomics-{GENE_ID}", "Metadata", "All"): df_genomics_info_parsed_all,
        # make_filename("REDS", "Index", f"Genomics-{GENE_ID}", "Alleles", "All"): df_alleles_index_all,
        make_filename("REDS", "Index", f"Genomics-{GENE_ID}", "Alleles"): df_alleles_index,
        make_filename("REDS", "Index", "Donor", f"Genomics-{GENE_ID}"): df_genomics_index,
        make_filename("REDS", "Recall", "Donor", f"Genomics-{GENE_ID}"): df_genomics_recall,
        # REDS Protein Metadata
        # make_filename("REDS", "Index", "RBC", "Proteomics", "Metadata"): df_proteomics_meta_index.set_index("UniProt"),
        # make_filename("REDS", "Recall", "RBC", "Proteomics", "Metadata"): df_proteomics_meta_recall.set_index("UniProt"),
        # REDS Donor Metadata
        make_filename("REDS", "Index", "Donor", "Metadata"): df_metadata_index,
        make_filename("REDS", "Recall", "Donor", "Metadata"): df_metadata_recall,
    }
)


output_dirpath /= "Formatted"
# Ensure output directory exists
output_dirpath.mkdir(parents=True, exist_ok=True)
output_filenames_dfs[output_dirpath].update(
    {
        # REDS Index
        make_filename("REDS", "Index", "Donor", "RBC", "Proteomics"): df_proteomics_index,
        make_filename("REDS", "Index", "Donor", "RBC", "Metabolomics"): df_metabolomics_index,
        # REDS Recall
        make_filename("REDS", "Recall", "Donor", "RBC", "Metabolomics"): df_metabolomics_recall,
        make_filename("REDS", "Recall", "Donor", "RBC", "Proteomics"): df_proteomics_recall,
        make_filename("REDS", "Recall", "Donor", "RBC", "Lipidomics"): df_lipidomics_recall,
        make_filename("REDS", "Recall", "Donor", "RBC", "TraceElements"): df_trace_elem_recall,
        # REDS Recall Lipidomics related
        make_filename(
            "REDS", "Recall", "Donor", "RBC", "Lipidomics", "DegUnsat"
        ): df_lipid_degunsat_recall,
        make_filename(
            "REDS", "Recall", "Donor", "RBC", "Lipidomics", "LipidClass"
        ): df_lipid_classes_recall,
        # REDS Recall Vesciles and mtDNA
        make_filename(
            "REDS", "Recall", "Donor", "RBC", "Vesicles", "mtDNA"
        ): df_vescicles_mtDNA_recall,
    }
)

for output_dirpath, output_filename_dfs in output_filenames_dfs.items():
    for output_filename, df in output_filename_dfs.items():
        save_dataframe_to_file(df, output_dirpath / output_filename, index=True)